# Imports

In [1]:
%load_ext autoreload
%autoreload 2

from transformers import RobertaForSequenceClassification
import torch
from torch.optim import AdamW
from transformers import get_scheduler, Trainer, TrainingArguments
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
import evaluate
from peft import LoraConfig, TaskType
from peft import get_peft_model
import huggingface_hub

# Load The Model

In [18]:
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8, 
    lora_alpha=32, 
    lora_dropout=0.1,
    target_modules="all-linear"
)

id2label = {0: "Bearish", 1: "Bullish"}
label2id = {"Bearish": 0, "Bullish": 1}

model = RobertaForSequenceClassification.from_pretrained('roberta-base', num_labels=2, id2label=id2label, label2id=label2id)
tokenizer = AutoTokenizer.from_pretrained('roberta-base')

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 1,919,234 || all params: 126,566,404 || trainable%: 1.5164


# Prepare Dataset

In [12]:
from datasets import Dataset
from transformers import DataCollatorWithPadding

pm_df = pd.read_parquet('../valid_comment_sentiment.parquet')

train_df, val_df = train_test_split(pm_df, test_size=0.2, random_state=42, stratify=pm_df['label'])

train_dataset = Dataset.from_pandas(train_df[['body', 'label']])
test_dataset = Dataset.from_pandas(val_df[['body', 'label']])

def tokenize_function(examples):
    return tokenizer(examples['body'], truncation=True, padding='max_length', max_length=256)

tokenized_datasets = {
    'train': train_dataset.map(tokenize_function, batched=True),
    'test': test_dataset.map(tokenize_function, batched=True)
}

tokenized_datasets['train'] = tokenized_datasets['train'].rename_column('label', 'labels')
tokenized_datasets['test'] = tokenized_datasets['test'].rename_column('label', 'labels')

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    metric = evaluate.combine(["accuracy", "f1", "precision", "recall"])
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return metric.compute(predictions=predictions, references=labels)


Map:   0%|          | 0/424678 [00:00<?, ? examples/s]

Map:   0%|          | 0/106170 [00:00<?, ? examples/s]

{'train': Dataset({
     features: ['body', 'labels', '__index_level_0__', 'input_ids', 'attention_mask'],
     num_rows: 424678
 }),
 'test': Dataset({
     features: ['body', 'labels', '__index_level_0__', 'input_ids', 'attention_mask'],
     num_rows: 106170
 })}

# Training Loop

In [16]:
account_name = huggingface_hub.whoami()['name']
training_args = TrainingArguments(
    output_dir=f"{account_name}/roberta-polymarket-sentiment-lora",
    learning_rate=1e-3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=True,
)

In [19]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

/Users/leventemurgas/Codes/DTU/02807_Comp_Tools/polymarket-computational-tools/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

# Evaluation

In [ ]:
metric = evaluate.combine(["accuracy", "f1", "precision", "recall"])
model.eval()

for batch in val_loader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)

    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)
    metric.add_batch(predictions=predictions, references=batch["labels"])

metric.compute()

{'accuracy': 0.94156,
 'f1': 0.9417812313209802,
 'precision': 0.9382294561333863,
 'recall': 0.94536}